# MCP 03 · 同一个 MCP 服务端，三种 Agent 怎么调

前两课我们把「服务端」和「客户端」分别跑通了。这一课是**横向对比**：
**服务端只起一个**（四则运算 + 天气查询），然后先后用三种方式去连它——

| # | 用什么调 | 对应源文件 |
|---|---|---|
| ① | **原生 OpenAI SDK**，手写 Function Call 循环 | `05_agent调用_openai.py`（原版 85 行）+ `_jxsd.py`（完整版 316 行） |
| ② | **LangChain**（`langchain-mcp-adapters` + `create_agent`） | `06_agent调用_langchain.py`（原版 75 行）+ `_jxsd.py`（完整版 264 行） |
| ③ | **DeepAgents**（`create_deep_agent`） | `07_agent调用_deepagents_jxsd.py`（314 行，无课案原版） |

> **本 notebook 由 `Agent/_py_source/05_mcp/` 下 5 个脚本合并而成**：
> `05_agent调用_openai.py`、`05_agent调用_openai_jxsd.py`、`06_agent调用_langchain.py`、
> `06_agent调用_langchain_jxsd.py`、`07_agent调用_deepagents_jxsd.py`。

## 一张表看懂「谁替你把 MCP 工具变成了 Agent 的工具」

| 问题 | ① 原生 OpenAI SDK | ② LangChain | ③ DeepAgents |
|---|---|---|---|
| **谁做工具格式转换** | **你自己**：手写 `mcp_tool_to_openai()`，把 `name/description/inputSchema` 拼成 `{"type": "function", ...}` | **`langchain-mcp-adapters`**：`MultiServerMCPClient(...).get_tools()` 一行转成 `BaseTool` | **同一个适配器，一字不差** |
| 谁跑工具循环 | **你自己**：`for _ in range(N)` + `if not msg.tool_calls: break` | `create_agent()` 内部的 LangGraph 图 | `create_deep_agent()` 内部 + 一整套内置中间件 |
| 工具怎么执行 | 你手动 `await client.call_tool(name, json.loads(args))` | 模型自己决定，框架执行 | 同左 |
| 要写的代码量 | ≈ 85 行（转换 + 循环 + 消息拼装） | 3 行：建 client → `get_tools()` → `create_agent()` | 3 行，但会**额外挂上一整套内置工具** |
| 你额外得到什么 | 什么都没有，但**全部机制都看得见** | 多服务混挂（stdio / http 混用）、工具名前缀 | 文件系统、任务拆解、子 Agent 委派 |
| 代价 | 换个框架就得重抄一遍 | 多一层依赖 | 每轮 tool schema 更长，**token 消耗明显更高** |
| 适用场景 | 最小依赖、教学、要精确控制每一步 | 绝大多数生产 Agent | 多步骤 + 需要留下中间产出（文件）的复杂任务 |

一句话结论：**MCP 把「工具」标准化了，适配器把「标准化」变成了「好用」；
而协议层与框架层一旦解耦，下游换成谁（LangChain / DeepAgents / 自研）都不用改那两行。**

**官方文档**
- MCP 协议官网：<https://modelcontextprotocol.io/>
- FastMCP 文档：<https://gofastmcp.com/>
- langchain-mcp-adapters：<https://github.com/langchain-ai/langchain-mcp-adapters>
- LangChain Agents：<https://docs.langchain.com/oss/python/langchain/agents>
- DeepAgents：<https://docs.langchain.com/oss/python/deepagents/overview>

## 运行条件

| 项 | 说明 |
|---|---|
| 🔴 运行档位 | **需外部服务** —— notebook 内**自己把 MCP 服务端起起来**（`127.0.0.1:8130`），跑完自己关掉 |
| 依赖 | `fastmcp` / `openai` / `langchain` / `langchain-mcp-adapters` / `deepagents`（venv 已装） |
| 密钥 | `settings.api_key` / `settings.base_url` / `settings.model_name`（已配置）—— 三种 Agent 都会**真实调用大模型** |
| 前置服务 | MCP 服务端由本 notebook 自己拉起，不需要另开窗口 |
| 端口 | **8130**（同章 5 个 notebook 并发，各占一个高位端口，不要用 8000） |
| 预计耗时 | 约 3~5 分钟（三种 Agent 合计约 10 轮模型调用） |

> `mcp.run(...)` 是**常驻服务**，直接在 cell 里跑会永久阻塞内核。所以本 notebook
> 走模板第 6 节第 5 条：把服务端写成一个能独立跑的 `server.py`，
> 用 `subprocess.Popen` 后台起进程，轮询端口就绪后再连它，
> **最后一个 cell** 用 `taskkill /T` 连子进程树一起收掉。

> 本机 Clash 会拦回环请求 —— 下面的自检格会确保把 `127.0.0.1` 加进 `NO_PROXY`。

## 本节地图

```mermaid
graph TD
    S["MCP 服务端 server.py :8130<br/>add/sub/mul/div + get_weather"]
    A["① 原生 OpenAI SDK<br/>手写 mcp_tool_to_openai() + 手写循环"]
    B["② LangChain<br/>MultiServerMCPClient.get_tools()"]
    C["③ DeepAgents<br/>同一个 get_tools() + create_deep_agent"]
    S -->|"HTTP streamable-http"| A
    S -->|"HTTP streamable-http"| B
    S -->|"HTTP streamable-http"| C
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 环节 | 发生在哪里 | 谁负责 |
|---|---|---|
| 起 MCP 服务端 | 独立进程 `server.py`，监听 `127.0.0.1:8130/mcp` | 本 notebook（`subprocess.Popen`） |
| ① 工具格式转换 | notebook 里的 `mcp_tool_to_openai()` | **你** |
| ① 工具循环 | notebook 里的 `for ... range(MAX_ROUNDS)` | **你** |
| ② / ③ 工具格式转换 | `MultiServerMCPClient.get_tools()` 内部 | **`langchain-mcp-adapters`** |
| ② / ③ 工具循环 | `create_agent` / `create_deep_agent` 编译出的 LangGraph 图 | **框架** |
| 关服务 | 最后一个 cell 的 `taskkill /F /T` | 本 notebook |

与上一节的衔接：`01_服务端.ipynb` 讲「怎么把普通函数变成 MCP 工具」，
这一节讲「工具变成 Agent 的工具之后，三种调用方式差在哪」。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 预期输出

```text
仓库根： F:\ProGram\Python_Base
临时目录： F:\ProGram\Python_Base\Agent\05_mcp\tmp_nb_work
```

两行都指向**仓库根**而不是 notebook 所在目录 —— 说明 `chdir` 生效了，
后面 `from config import settings` 才找得到 `config.py`。

### 0.1 前置条件自检

三件事，缺一件后面都会以报错收场，所以先查一遍再往下走：

1. **`NO_PROXY` 必须含 `127.0.0.1`** —— 本机 Clash 会把回环请求拦走，
   表现为「服务明明起来了却连接被拒 / 502」；
2. **五个依赖包**是否都在（`fastmcp` / `openai` / `langchain` / `langchain-mcp-adapters` / `deepagents`）；
3. **模型配置**是否读到了。

In [ ]:
import importlib
import socket

# 本机 Clash 拦回环：把自己要用的地址补进 NO_PROXY（已经设过就不动）
for _key in ("NO_PROXY", "no_proxy"):
    _val = os.environ.get(_key, "")
    if "127.0.0.1" not in _val:
        os.environ[_key] = (_val + "," if _val else "") + "127.0.0.1,localhost"

NEEDED = ["fastmcp", "openai", "langchain", "langchain_mcp_adapters", "deepagents"]
_missing = [name for name in NEEDED if importlib.util.find_spec(name) is None]
print("NO_PROXY =", os.environ.get("NO_PROXY"))
if _missing:
    print("❌ 缺少依赖：", _missing)
    print("   安装命令： uv add " + " ".join(_missing))
else:
    print("✅ 依赖齐全：", NEEDED)

from config import settings

print("模型：", settings.model_name)
print("接口：", settings.base_url)
if not settings.api_key:
    print("❌ settings.api_key 为空 —— 配置好仓库根的 .env 后再重跑本 notebook")
else:
    print("✅ settings.api_key 已配置（值不回显）")

### 预期输出

```text
NO_PROXY = 127.0.0.1,localhost
✅ 依赖齐全： ['fastmcp', 'openai', 'langchain', 'langchain_mcp_adapters', 'deepagents']
模型： deepseek-flash
接口： https://api.deepseek.com
✅ settings.api_key 已配置（值不回显）
```

## 1. 起服务（上）：课案原版的**进程内后台线程**

三个 `_jxsd.py` 源文件都带一个 `start_server_in_thread()`。它做的事是：
用 `importlib` 按文件路径加载一个服务端脚本（文件名以数字开头没法 `import`），
拿到它的 `mcp` 实例，再用**后台线程**跑 uvicorn。

为什么不能直接 `mcp.run(transport="streamable-http", ...)`？因为它内部是一个
uvicorn 事件循环，会**永久占住当前线程** —— 在 notebook 里就是「这一格永远跑不完」。

### 1.1 服务端的工具从哪来：**原地定义，不 import 归档脚本**

源文件的写法是 `importlib` 加载归档的 `01_服务端_jxsd.py`，把它的工具「借」过来，
只补一个 `get_weather`。本 notebook **改成把 5 个工具在 `server.py` 里原地定义**，
原因是一条实测踩到的坑（详见「常见坑」第 1 条）：

> 归档脚本是按「**独立可执行**」写的，文件头就有
> `sys.stdout.reconfigure(encoding="utf-8")`。在命令行/子进程里没问题，
> 但**一旦被 notebook 内核 in-process 加载**，`sys.stdout` 是 IPython 的 `OutStream`，
> 没有 `reconfigure` 方法 → 直接
> `AttributeError: 'OutStream' object has no attribute 'reconfigure'`。

所以本课的规矩是：**归档脚本只当子进程跑，不要在 notebook 进程里 import 它。**
这样做还有一个好处：5 个工具在 `server.py` 里原样展开，
读者一眼就能看到「工具 = 普通函数 + 类型注解 + docstring」——
这正是第 01 课的核心，在这里再看一次。

> `server.py` 写在 `WORKDIR/mcp_agent_call/` 下（`WORKDIR` 是同章共享的，
> 不套一层自己的目录，同章并发执行时会互相踩文件）。

In [ ]:
import asyncio
import json
import subprocess
import threading
import time

HTTP_HOST = "127.0.0.1"
HTTP_PORT = 8130                      # 课案默认 8000；本 notebook 改用 8130（同章并发防撞端口）
MCP_URL = f"http://{HTTP_HOST}:{HTTP_PORT}/mcp"

# 本课专属子目录：WORKDIR（tmp_nb_work）是**同章共享**的，
# 不套一层自己的目录，同章并发执行时会互相踩文件。
SERVER_DIR = WORKDIR / "mcp_agent_call"
SERVER_DIR.mkdir(parents=True, exist_ok=True)

# 按文件路径加载，所以 SERVER_SCRIPT 就是一个普通变量（源文件里写的是 Path(__file__)... ）
SERVER_SCRIPT = SERVER_DIR / "server.py"

SERVER_SOURCE = '''# -*- coding: utf-8 -*-
"""MCP 服务端（由 notebook 03_三种Agent调用.ipynb 生成）：四则运算 + 天气查询。

⚠️ 这里**故意不 import 归档区的 01_服务端_jxsd.py**，而是把工具原地定义一份。
原因（本机实测踩到的坑）：那些归档脚本是按「**独立可执行**」写的，文件头就有
    sys.stdout.reconfigure(encoding="utf-8")
在命令行/子进程里跑没问题，但**一旦被 notebook 内核 in-process 加载**，
sys.stdout 是 IPython 的 OutStream、根本没有 reconfigure 方法 —— 直接
    AttributeError: 'OutStream' object has no attribute 'reconfigure'。
（同一族问题还有：mcp 的 stdio 客户端把 sys.stderr 当默认参数在导入时绑定，
OutStream 没有真 fileno() → RuntimeError: Client failed to connect: fileno。）
所以规矩是：**归档脚本只当子进程跑，不要在 notebook 进程里 import 它。**

顺带好处：工具在这里原样展开，读者一眼就能看到「工具 = 普通函数 + 类型注解 + docstring」。

单独运行：
    python server.py            # streamable-http，监听 127.0.0.1:8130/mcp
    python server.py stdio      # 交给 MCP 客户端当子进程拉起
"""
import logging
import sys

from fastmcp import FastMCP

# 注意是**fastmcp 这个 logger**：设 root 级别无效（fastmcp 用自己的 handler 且不向 root 传播）
logging.getLogger("fastmcp").setLevel(logging.ERROR)

mcp = FastMCP("演示 🚀")


# 类型注解 → 参数 JSON Schema；docstring → 工具的 description（模型靠它选工具）
@mcp.tool
def add(a: float, b: float) -> float:
    """两数相加
    :param a:第一个数字
    :param b:第二个数字
    :return: a+b的结果
    """
    return a + b


@mcp.tool
def sub(a: float, b: float) -> float:
    """两数相减
    :param a:第一个数字
    :param b:第二个数字
    :return: a-b的结果
    """
    return a - b


@mcp.tool
def mul(a: float, b: float) -> float:
    """两数相乘
    :param a:第一个数字
    :param b:第二个数字
    :return: a*b的结果
    """
    return a * b


@mcp.tool
def div(a: float, b: float) -> float:
    """两数相除
    :param a:第一个数字
    :param b:第二个数字
    :return: a/b的结果
    """
    # 除零是工具里唯一的业务异常：MCP 会把它包成 is_error=True 的结果回给客户端，
    # 而不是让整条连接崩掉，模型看到错误信息后可以自己纠正参数重试。
    if b == 0:
        raise ValueError("除数不能为 0")
    return a / b


@mcp.tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气。city：城市名称，如「上海」"""
    weather_map = {"上海": "晴 25 度", "北京": "多云 18 度", "广州": "阵雨 30 度"}
    return weather_map.get(city, f"{city} 天气未知")


HTTP_HOST = "127.0.0.1"
HTTP_PORT = 8130

if __name__ == "__main__":
    import uvicorn

    if len(sys.argv) > 1 and sys.argv[1] == "stdio":
        mcp.run(transport="stdio")
    else:
        app = mcp.http_app(transport="streamable-http", path="/mcp")
        uvicorn.run(app, host=HTTP_HOST, port=HTTP_PORT, log_level="warning")
'''

SERVER_SCRIPT.write_text(SERVER_SOURCE, encoding="utf-8")
print("服务端脚本已写入：", SERVER_SCRIPT.relative_to(ROOT), f"（{len(SERVER_SOURCE.splitlines())} 行）")


def _port_in_use(host: str, port: int) -> bool:
    """能连上就说明端口已被监听 —— 用来判断服务端是不是已经起来了。"""
    # 短超时探测：连不上只说明没人监听，不算错误，不能让它卡住。
    sock = socket.socket()
    sock.settimeout(0.5)
    try:
        sock.connect((host, port))
        return True
    except OSError:
        return False
    finally:
        sock.close()


def start_server_in_thread():
    """后台线程起 MCP 服务端；端口被占就直接连已有的。

    返回 None 表示「不是我起的」，退出时就不能去关它 —— 那可能是别人起的服务端。
    """
    if _port_in_use(HTTP_HOST, HTTP_PORT):
        print(f"ℹ️  {HTTP_HOST}:{HTTP_PORT} 已有 MCP 服务在运行，直接连它。")
        return None

    # 动态加载 SERVER_SCRIPT：文件名以数字开头没法 import，
    # 用 importlib 从文件路径加载也不会触发它 __main__ 里的启动分支。
    import importlib.util

    import uvicorn

    spec = importlib.util.spec_from_file_location("mcp_server_01", SERVER_SCRIPT)
    module = importlib.util.module_from_spec(spec)
    sys.modules["mcp_server_01"] = module
    spec.loader.exec_module(module)

    app = module.mcp.http_app(transport="streamable-http", path="/mcp")
    server = uvicorn.Server(
        uvicorn.Config(app, host=HTTP_HOST, port=HTTP_PORT, log_level="warning")
    )
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(100):
        if server.started:
            return server, thread
        time.sleep(0.1)
    print(f"❌ MCP 服务端启动失败：{HTTP_HOST}:{HTTP_PORT} 无法监听。")
    return None


# 起它一次，确认归档里那套「进程内线程」写法在本机确实能用
started = start_server_in_thread()
print("进程内线程服务端：", "已启动" if started else "没启动（端口已被占）")

### 预期输出

```text
服务端脚本已写入： Agent\05_mcp\tmp_nb_work\mcp_agent_call\server.py （93 行）
进程内线程服务端： 已启动
```

两件事同时被证实：**`server.py` 真的落盘了**（`WORKDIR/mcp_agent_call/` 下），
**后台线程的 uvicorn 真的起来了**。注意这一格**没有卡住** ——
这正是「不能直接写 `mcp.run(...)`」的反面证据：
换成 `mcp.run(...)`，这一格就再也回不来了。

### 1.2 进程内线程的代价：三条

1. **服务和 notebook 同生共死** —— 内核一重启，服务就没了；
   而且它跑在 `daemon=True` 的线程里，内核退出时**不会**打印任何收尾信息；
2. 它得靠 `importlib` 在**当前进程里再加载一遍**另一个 `.py`（连同 pydantic 模型
   一起进 `sys.modules`），路径一改就散架；
3. 它和真实部署差得最远：真实部署是**独立进程 + 端口**，
   别的客户端（Claude Desktop / 别的语言）也能连上来。

所以本 notebook 后面统一改成 **`subprocess.Popen` 起独立进程**（下一格），
并且把服务端写成能自己单独跑的 `server.py`：想看它独立运行，直接 `python server.py`。

In [ ]:
# 关掉上面那个进程内服务端：先通知 uvicorn 退出，再等线程真的结束。
# 顺序反了会打印到一半日志就断掉，而且端口可能还没释放。
if started:
    server, thread = started
    server.should_exit = True
    thread.join(timeout=10)
    # 等端口真的空出来再往下（Windows 上刚关闭的监听套接字不一定立刻可重绑）
    for _ in range(40):
        if not _port_in_use(HTTP_HOST, HTTP_PORT):
            break
        time.sleep(0.25)
    print("进程内线程服务端已关闭，端口已释放：", not _port_in_use(HTTP_HOST, HTTP_PORT))

### 预期输出

```text
进程内线程服务端已关闭，端口已释放： True
```

`True` 是这一格的重点：**端口必须先空出来**，下一格才能用同一个 8130 起独立进程。
这也是「起服务」类 notebook 最容易翻车的地方 —— 上一格没关干净，
下一格要么起不来，要么连到了上一次的僵尸实例。

## 2. 起服务（下）：本 notebook 采用的**独立进程**

换成本 notebook 的方式：`subprocess.Popen` 起一个**独立进程**，
然后用 `_port_in_use()` 轮询等它监听成功（**不要把 `sleep` 写死太久**，
也不要假设它一定成功）。`poll() is None` 就说明子进程还活着。

这个进程会一直服务到 notebook 结束，**最后一个 cell** 用
`taskkill /F /T /PID` 连子进程树一起收掉。

In [ ]:
# 端口上已经有服务（可能是上一次没关干净，也可能是同章另一个 notebook 抢了 8130）
if _port_in_use(HTTP_HOST, HTTP_PORT):
    print(f"ℹ️  {HTTP_HOST}:{HTTP_PORT} 已有服务在监听 —— 可能不是我们自己的，"
          "下一格的「身份核对」会验明正身。")

env = {**os.environ, "PYTHONUTF8": "1", "NO_PROXY": "127.0.0.1,localhost"}
server = subprocess.Popen(
    [sys.executable, str(WORKDIR / "mcp_agent_call" / "server.py")],
    cwd=str(ROOT), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
)

# 轮询等就绪（不要把 sleep 写死太久，也不要假设它一定成功）
_ready = False
for _ in range(60):
    if _port_in_use(HTTP_HOST, HTTP_PORT):
        _ready = True
        break
    time.sleep(0.5)

print(f"MCP 服务端就绪：{_ready}   PID：{server.pid}   poll：{server.poll()}")
if server.poll() is not None:
    # 子进程已经退出：几乎一定是没抢到端口。把它自己的报错读出来 —— 比
    # 一句「连不上」有用得多（这也解释了为什么 stdout 要用 PIPE 接住）。
    print("❌ 子进程已经退出（多半是 8130 没抢到），它的输出如下：")
    print(server.stdout.read() if server.stdout else "(无输出)")
elif not _ready:
    print("❌ 等不到端口就绪（服务进程还活着，但没在监听）。")

### 预期输出

```text
MCP 服务端就绪：True   PID：50392   poll：None
```

`PID：` 后面那串数字每次都不同（它是进程号），**只有 `poll：None` 是判断依据** ——
它说明子进程还活着在监听。若打出 `poll：1`，通常是 8130 被别的程序占了，
下一格就会把子进程的输出打出来给你看。

## 3. 公共部分：模型客户端 + 先认识这个服务端

三种方式**共用**同一个服务端、同一套模型配置，差别只在「谁来转换、谁来跑循环」。
先把公共部分拿出来：

- `llm`：LangChain 侧的统一模型入口（`init_chat_model`，换厂商只改 `model_provider` 一个字符串）；
- `OpenAI` / `AsyncOpenAI`：原生 SDK 侧的两个客户端（课案原版用同步的，完整版用异步的）；
- `MAX_ROUNDS`：循环上限 —— 模型反复调工具时由它兜住，否则出 bug 会无限转下去烧钱。

In [ ]:
from openai import AsyncOpenAI
from openai import OpenAI
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient

client_llm = OpenAI(api_key=settings.api_key, base_url=settings.base_url)
openai_client = AsyncOpenAI(api_key=settings.api_key, base_url=settings.base_url)
MODEL = settings.model_name
MAX_ROUNDS = 8
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

# 【notebook 改写】5 个源文件的末尾都是：
#     if __name__ == "__main__":
#         asyncio.run(main())
# notebook 内核里**已经有事件循环**，再调 asyncio.run() 会 RuntimeError，
# 所以本 notebook 统一改成 **顶层 await**（IPython 的 autoawait 支持，本机已实测）。
# 下面这行只为「把本 notebook 导出成 .py 单跑」保留 —— 那个场景下 __file__ 才存在：
if "__file__" in globals():
    asyncio.run(main())

print("模型客户端就绪：", settings.model_name)

### 预期输出

```text
模型客户端就绪： deepseek-flash
```

这一格同时定义了一个很关键的东西：`MAX_ROUNDS = 8` 与那个
`if "__file__" in globals():` 守卫（见代码里的注释）。

先用最朴素的 `fastmcp.Client` 连一次，确认服务端真的活着、工具真的注册上了。
这一步**不经过任何 Agent 框架** —— 它是后面三种方式共同的底座。

> `Client(MCP_URL)` 给 URL 就够了：FastMCP 会按 URL 自动选出 streamable-http 传输。

In [ ]:
from fastmcp import Client

EXPECTED_TOOLS = sorted(["add", "sub", "mul", "div", "get_weather"])


async def probe():
    """裸 fastmcp 客户端：list_tools 看工具清单，call_tool 验一次真实调用。

    起服务最容易「看起来成了、其实连的是别人」—— 所以这里加一道**身份核对**：
    8130 上如果趴着别的 notebook 的服务端，**端口探测是看不出来的**
    （探测只知道「有人监听」），只有把工具清单拿出来比一比才知道。
    """
    async with Client(MCP_URL) as client:
        mcp_tools = await client.list_tools()
        print("服务端暴露的工具：", [t.name for t in mcp_tools])
        if sorted(t.name for t in mcp_tools) == EXPECTED_TOOLS:
            print("✅ 身份核对通过：这就是本课的服务端")
        else:
            print("❌ 8130 上趴着的不是本课的服务端（期望", EXPECTED_TOOLS, "）")
        result = await client.call_tool("add", {"a": 3, "b": 5})
        print("调用 add(3, 5) =", result.content[0].text)


# 连不上时不要只丢一个 "Client failed to connect: "，把能查的都打出来 ——
# 同章 5 个 notebook 并发时，最常见的原因就是 8130 被另一个 notebook 占了。
try:
    await probe()
except Exception as exc:
    print("❌ 连不上 MCP 服务端：", type(exc).__name__, exc)
    print("   ① 8130 是不是被占了？看第 2 节 `_port_in_use()` 那一行的提示；")
    print("   ② 我们起的子进程还活着吗？poll =", server.poll())
    if server.poll() is not None and server.stdout:
        print("   子进程输出：", server.stdout.read()[:1500])
    raise RuntimeError("MCP 服务端不可用 —— 三种调用都会失败，先按上面的提示排掉。")

### 预期输出

```text
服务端暴露的工具： ['add', 'sub', 'mul', 'div', 'get_weather']
✅ 身份核对通过：这就是本课的服务端
调用 add(3, 5) = 8.0
```

三行里最重要的是第二行：**端口通只说明「有人监听」，不代表「是自己人」**。
同章 5 个 notebook 并发时，只要有一个抢了 8130，后面这个 notebook 就会
「连上了、但工具对不上」——那句 `身份核对通过` 就是专门用来识破这种事的。

第三行 `8.0` 里的浮点是另一条线索：签名写的是 `def add(a: float, b: float)`，
注解写 `float` 就真按浮点走。**类型注解既决定给模型看的 JSON Schema，
也决定返回值长什么样** —— 这是第 01 课讲过的点，在这里再看一次。

工具顺序 = `server.py` 里的定义顺序：`add` / `sub` / `mul` / `div`，最后是 `get_weather`。
顺序本身不影响模型选工具（模型靠每个工具的 `description`，也就是 docstring 来选）。

**服务端就绪，下面开始三种调法。**

## 4. ① 原生 OpenAI SDK · 课案原版（85 行）

不用任何 Agent 框架，整条链路只有四步（就是 `05_agent调用_openai.py` 文件头写的那四步）：

    1. MCP 客户端 list_tools 拿到工具定义
    2. 转成 OpenAI 的 tools 格式（type=function / name / description / parameters）
    3. 模型返回 tool_calls 后，用 MCP 客户端 call_tool 执行
    4. 结果以 role="tool" 回填进 messages，再问一次模型

第 2 步是这一节的灵魂，就三行：

| MCP 侧字段 | OpenAI 侧字段 | 说明 |
|---|---|---|
| `tool.name` | `function.name` | 模型调用时回传的名字，**必须逐字一致** |
| `tool.description` | `function.description` | 来自服务端 docstring，**模型靠它选工具** |
| `tool.inputSchema` | `function.parameters` | 来自服务端类型注解生成的 JSON Schema |

反过来，模型回的结果也要转回去：

    模型的 tool_calls[i] → client.call_tool(name, json.loads(arguments))
    工具结果            → {"role": "tool", "tool_call_id": ..., "content": 文本}

**框架帮你省的就是这来回两次转换。** 第 6 节（LangChain）会看到它被压缩成两行。

In [ ]:
def mcp_tool_to_openai(tool) -> dict:
    """把 MCP 工具定义转换成 OpenAI Function Call 描述格式"""
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            # inputSchema 已经是标准 JSON Schema，两边格式完全一致，直接搬
            "parameters": tool.inputSchema,
        },
    }


# 课案原版走的是 **stdio 传输**（客户端把服务端当子进程拉起来）。
# 本 notebook 改走 **HTTP** —— 两个原因，都是本机实测出来的（见「常见坑」第 2、3 条）：
#   ① Jupyter 内核在 Windows 上跑的是 SelectorEventLoop，**不支持 asyncio 子进程**，
#      所以 StdioTransport 在 notebook 里根本连不上（报 "Client failed to connect: fileno"）；
#   ② 就算在普通脚本里 stdio 能用，`create_deep_agent` 配 stdio 会**卡死**，HTTP 才正常。
# 两种传输的构造对照放在这里 —— 第一行只用来说明课案的写法，本 notebook 不连它：
from fastmcp.client.transports import StdioTransport
from fastmcp.client.transports import StreamableHttpTransport

transport_stdio = StdioTransport(command=sys.executable, args=[str(SERVER_SCRIPT), "stdio"])
transport = StreamableHttpTransport(MCP_URL)

下面是课案原版的主体：`for _ in range(10)` 的 Function Call 循环。

三个细节值得停一下：

- `async with Client(transport) as mcp:` —— 直接给 `transport` 而不是 URL，
  本 notebook 传进来的是 HTTP 传输（课案传的是 stdio 传输，客户端侧写法**一模一样**）；
- `messages.append(msg.model_dump())` —— 用 SDK 的 `model_dump()` 而不是手写 dict，
  这样 `refusal` / `annotations` 这些字段会一并带上，避免下一轮 400；
- `role="tool"` 的消息**必须带 `tool_call_id`**，模型靠它把结果和请求对上。

> 课案原版问的是「上海天气怎么样？」—— 服务端的 `get_weather` 就是为它准备的。

In [ ]:
async def main():
    async with Client(transport) as mcp:
        # 1. 拉取 MCP 工具并转换格式
        mcp_tools = await mcp.list_tools()
        openai_tools = [mcp_tool_to_openai(t) for t in mcp_tools]
        print("接入的 MCP 工具：", [t.name for t in mcp_tools])

        messages = [
            {"role": "system", "content": "你是生活助手，优先使用工具回答。"},
            {"role": "user", "content": "上海天气怎么样？"},
        ]

        # 2. Function Call 循环
        for _ in range(10):
            resp = client_llm.chat.completions.create(
                model=settings.model_name,
                messages=messages,
                tools=openai_tools,
            )
            msg = resp.choices[0].message
            messages.append(msg.model_dump())

            if not msg.tool_calls:
                print("AI：", msg.content)
                break

            for call in msg.tool_calls:
                # 3. 用 MCP 客户端执行工具（不再本地找函数）
                result = await mcp.call_tool(
                    call.function.name,
                    json.loads(call.function.arguments),
                )
                print(f"[MCP 工具 {call.function.name}] {result}")
                # 4. 结果回填模型
                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": str(result),
                })


await main()

### 预期输出

```text
接入的 MCP 工具： ['add', 'sub', 'mul', 'div', 'get_weather']
```

这一行就是**步骤 1**（`list_tools`）：原生 SDK 侧拿到的工具清单，
和上一格用裸 `fastmcp.Client` 看到的一模一样 —— 因为**转换只改形状，不改内容和数量**。

### 预期输出（后面两行由模型决定，每次运行不同）

```text
[MCP 工具 get_weather] CallToolResult(content=[TextContent(type='text', text='晴 25 度',
    annotations=None, meta=None)], structured_content={'result': '晴 25 度'},
    meta={'fastmcp': {'wrap_result': True}}, data='晴 25 度', is_error=False)
AI： 上海目前天气**晴**，气温 **25℃** 🌞

天气舒适宜人，适合外出活动。不过晴天紫外线可能较强，建议出门时做好防晒……
```

第二行 = **步骤 3**（模型发了 `tool_calls`，我们转手交给 MCP 客户端执行）；
第三行 = **步骤 4**（结果回填后再问一次，模型这次不再要工具，循环正常退出）。
模型说了几句、用词如何，**由模型决定**，别逐字比对。

真正值得盯的是第二行的**形状**：`str(result)` 打出来是一个 `CallToolResult(...)` 对象，
里面套着 `content=[TextContent(type='text', text='晴 25 度', ...)]`。
**这就是「手工转换」的真实手感：连结果的形状也得你自己处理。**
完整版会把它取成纯文本（`result.content[0].text`），
LangChain 版则统一成 `ToolMessage.content` —— 框架替你把这一层磨平了。

## 5. ① 原生 SDK · 完整版（316 行）：给循环补上工程护栏

完整版比原版多了三样东西，都是「让 Agent 能被信任」的关键：

| 补了什么 | 为什么 |
|---|---|
| `AsyncOpenAI` + `await` | 整个链路在异步环境里，await 调用更自然，不阻塞事件循环 |
| `MAX_ROUNDS` 上限 | 模型陷入「反复调工具」时由 `for` 兜住，**不会无限烧钱** |
| `count_operations()` 流程完整性校验 | 见下 —— 这是本课最值得带走的一条工程经验 |

### 5.1 模型「偷懒少调工具」是真实存在的

`05_agent调用_openai_jxsd.py` 的文件头记录了一条本机实测：

- 「8\*2-9」有时只调 `mul` 就答 16；
- 「2+4\*6」有时只调 `mul` 就答 24，甚至编出工具从没返回过的 48。

光靠系统提示词「请务必逐步调用工具」是压不住的。正确做法是
**用程序算出流程下界，再检查实际执行有没有达到** ——
与其祈祷模型自觉，不如让代码在它偷懒时把它推回去。
这就是 Agent 工程里说的「流程完整性校验 / guardrail」。

In [ ]:
def count_operations(expr: str) -> int:
    """数一数表达式里有几个二元运算符，作为「至少要调用几次工具」的下界。

    这个简陋的实现只适合加减乘除的纯表达式（负数、括号会数错），
    生产里应该换成真正的表达式解析器。
    """
    return sum(expr.count(op) for op in "+-*/")


for question in ["8*2-9", "2+4*6"]:
    print(f"{question} 至少需要 {count_operations(question)} 步运算")

### 预期输出

```text
8*2-9 至少需要 2 步运算
2+4*6 至少需要 2 步运算
```

两条都是 2 步，但**优先级不同**：`8*2-9` 是「先乘后减」，
`2+4*6` 是「先乘后加」。这个下界只数运算符个数，
所以它回答的是「至少几步」而不是「哪几步」—— 具体拆成哪几步仍要看模型怎么选。

### 5.2 完整版的循环：转换 → 调用 → 回填 → 校验

和原版相比，循环体里多出两处「护栏」：

- 收尾前先做**流程完整性校验**：如果 `tool_calls_made < required_steps`，
  就补一条 `user` 消息把模型推回去继续算（最多推 2 次，避免死活不改时无限重试）；
  ⚠️ 推回去的关键是**先把助手那句话原样记进历史**（保持对话结构完整）再催它 ——
  直接改 system 提示词没用，模型只看「最近发生了什么」；
- `for ... else`：循环跑满 `MAX_ROUNDS` 还没 `break`，就是陷入工具循环了，明确中止。

In [ ]:
async def main() -> None:
    client = Client(MCP_URL)

    async with client:
        # ---------- 步骤 1：拉取 MCP 工具定义 ----------
        mcp_tools = await client.list_tools()
        print("加载MCP工具：", [t.name for t in mcp_tools])

        # ---------- 步骤 2：转成 OpenAI 的 tools 格式 ----------
        openai_tools = [mcp_tool_to_openai(t) for t in mcp_tools]
        # 把转出来的结构打出来看一眼，这就是「框架替你做的那件事」长什么样
        print("转换后的 OpenAI 工具格式（以第一个为例）：")
        print(json.dumps(openai_tools[0], ensure_ascii=False, indent=2))

        # ---------- 步骤 3 & 4：真实的 Function Call 循环 ----------
        # 「8*2-9」和「2+4*6」都是两步运算，但优先级不同，
        # 两条题各跑一遍能看出模型是不是真的按顺序调工具。
        for question in ["8*2-9", "2+4*6"]:
            print("\n" + "=" * 60)
            print(f"用户：{question}")
            print("=" * 60)

            messages = [
                {
                    "role": "system",
                    # 系统提示词定规矩：**每一步**算术都要交给工具，禁止心算。
                    # 实测不写「每一步」，模型经常只调一次 mul 就把 2+24 心算掉，甚至算错。
                    "content": (
                        "你是一个助手，可以帮助用户进行数学计算。严格遵守以下规则：\n"
                        "1. 任何一次算术运算都必须调用工具完成，严禁心算或直接给出数字；\n"
                        "2. 表达式里有几步运算，就分几步调用工具，"
                        "前一步工具的结果作为下一步工具的输入；\n"
                        "3. 所有步骤都算完后，最后一行按「最终结果：<数字>」的格式输出；\n"
                        "4. <数字> 必须是工具返回过的值，不得自行编造。"
                    ),
                },
                {"role": "user", "content": question},
            ]

            # 流程下界：这个表达式至少要几次工具调用才算算完
            required_steps = count_operations(question)
            tool_calls_made = 0     # 实际调用了多少次
            push_backs = 0          # 因为「少调了工具」把模型推回去的次数

            # MAX_ROUNDS 是硬保险：模型陷入工具循环时由这个 for 兜住。
            for round_no in range(1, MAX_ROUNDS + 1):
                response = await openai_client.chat.completions.create(
                    model=MODEL,
                    messages=messages,
                    tools=openai_tools,
                    tool_choice="auto",
                )
                msg = response.choices[0].message

                # 没有 tool_calls 说明模型认为可以收尾了 —— 这是循环的常规退出条件
                if not msg.tool_calls:
                    # 但收尾之前先做「流程完整性校验」：少调了工具就推回去补齐
                    if tool_calls_made < required_steps and push_backs < 2:
                        push_backs += 1
                        print(
                            f"⚠️  模型只调用了 {tool_calls_made} 次工具，但 {question} "
                            f"至少需要 {required_steps} 步运算 —— 已把它推回去补齐（第 {push_backs} 次）。"
                        )
                        messages.append({"role": "assistant", "content": msg.content or ""})
                        messages.append({
                            "role": "user",
                            "content": (
                                f"你只调用了 {tool_calls_made} 次工具，还有运算没有算完"
                                f"（{question} 至少需要 {required_steps} 步）。"
                                "请继续调用工具算出剩余步骤，禁止心算，算完后按"
                                "「最终结果：<数字>」输出。"
                            ),
                        })
                        continue

                    print(f"\n最终回答：{msg.content}")
                    break

                # 把 assistant 这条（含 tool_calls）原样追加进历史。
                # 用 model_dump() 而不是手写 dict：SDK 的新字段它会一并带上，避免漏字段 400。
                messages.append(msg.model_dump())

                for tool_call in msg.tool_calls:
                    func_name = tool_call.function.name
                    # arguments 是**字符串**（模型吐的是 JSON 文本），必须 json.loads
                    func_args = json.loads(tool_call.function.arguments)
                    print(f"[第 {round_no} 轮] 调用MCP工具：{func_name}，参数：{func_args}")

                    # 关键一行：工具不在本地，而是通过 MCP 客户端远程执行
                    tool_result = await client.call_tool(func_name, func_args)
                    func_response = (
                        tool_result.content[0].text if tool_result.content else str(tool_result)
                    )
                    print(f"           工具返回结果：{func_response}")
                    tool_calls_made += 1

                    # role="tool" 的消息必须带 tool_call_id，模型靠它把结果和请求对上
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": func_name,
                        "content": func_response,
                    })
            else:
                # for-else：循环跑满 MAX_ROUNDS 还没 break，说明模型陷入工具循环了
                print(f"⚠️  已达最大轮次 {MAX_ROUNDS}，仍未给出最终回答，已中止。")


await main()

### 预期输出（模型措辞与轮次每次运行不同）

```text
加载MCP工具： ['add', 'sub', 'mul', 'div', 'get_weather']
转换后的 OpenAI 工具格式（以第一个为例）：
{
  "type": "function",
  "function": {
    "name": "add",
    "description": "两数相加",
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "a": { "type": "number", "description": "第一个数字" },
        "b": { "type": "number", "description": "第二个数字" }
      },
      "required": [ "a", "b" ],
      "type": "object"
    }
  }
}

============================================================
用户：8*2-9
============================================================
[第 1 轮] 调用MCP工具：mul，参数：{'a': 8, 'b': 2}
           工具返回结果：16.0
[第 2 轮] 调用MCP工具：sub，参数：{'a': 16, 'b': 9}
           工具返回结果：7.0

最终回答：最终结果：7

============================================================
用户：2+4*6
============================================================
[第 1 轮] 调用MCP工具：mul，参数：{'a': 4, 'b': 6}
           工具返回结果：24.0
[第 2 轮] 调用MCP工具：add，参数：{'a': 2, 'b': 24}
           工具返回结果：26.0

最终回答：先算乘法：4×6=24
再算加法：2+24=26

最终结果：26
```

这一格是本课信息量最大的一格，四个点按顺序看：

1. **`转换后的 OpenAI 工具格式`** —— 打印出来的就是「框架背地里替你做的那件事」：
   `description` 来自服务端 docstring，`parameters` 来自类型注解生成的
   `additionalProperties: false` + `required: [a, b]` 的 JSON Schema。
   LangChain 那一节的 `args_schema` 就是同一份东西换了层皮。
2. **两条题都是 2 步，且真的分 2 轮调完** —— 说明系统提示词里
   「表达式里有几步运算，就分几步调用工具」生效了。
3. **工具返回 `16.0` / `7.0` 这种浮点**，而模型最后输出的是整数 `7` / `26`
   （它按 `「最终结果：<数字>」` 的格式要求做了收敛）。
4. **本次两条题都一次通过，没有出现「推回去」那行警告** ——
   但那条分支是必须留着的：`05_agent调用_openai_jxsd.py` 的文件头记录了
   「只调一次 `mul` 就把 `2+4*6` 答成 24」的实测。**护栏平时不响，不代表可以省掉。**

轮次与用词由模型决定，所以这段是「结构一致、细节可能不同」：
真正的判据是「每条表达式都出现对应步数的工具调用」。

## 6. ② LangChain · 课案原版（75 行）：两行把 MCP 变成 BaseTool

上一节手写了「MCP 工具 ↔ OpenAI tools 格式」来回转换的循环。
这一节用 `langchain-mcp-adapters`，把整个循环压缩成两行：

    client = MultiServerMCPClient({...})    # 描述连哪些 MCP 服务
    tools = await client.get_tools()        # 自动转换格式（等价于上一节的手工转换）

之后直接 `create_agent(model=llm, tools=tools)` —— **中间那句转换代码从此不用写了**。

| 对比项 | ① 原生 SDK | ② langchain-mcp-adapters |
|---|---|---|
| 工具格式转换 | 手写 `mcp_tool_to_openai()` | `get_tools()` 自动完成 |
| 执行循环 | 手写 `for` + `tool_calls` 判断 | Agent 内部完成 |
| 多服务混挂 | 自己管多个 `Client` | 一个 dict 配多个服务（stdio/http 混用） |
| 调用方式 | `await client.call_tool()` | 模型自动决定 |

### 6.1 版本坑：不要用 `async with`

`langchain-mcp-adapters ≥ 0.1.0`（本机 0.3.2）**不支持**
`async with MultiServerMCPClient(...)`，会抛 `NotImplementedError`。
正确写法就是**直接实例化**然后 `await client.get_tools()`。

### 6.2 课案原版用的是 stdio 配置

下面这个 dict 是课案原版的写法（`"command": "uv"` + `args`，客户端把服务端
当**子进程**拉起来）。本 notebook **不连它**，只留作对照 —— 原因就是第 4 节说的那两条。

In [ ]:
from deepagents import create_deep_agent

# 课案原版的 stdio 配置（客户端自己拉起服务端子进程，不占端口）。
# 本 notebook 只把它打印出来做对照，不拿它建连接：
STDIO_CONFIG = {
    "life": {
        "command": "uv",
        "args": ["run", str(SERVER_SCRIPT), "stdio"],
        "transport": "stdio",
    },
}
print("课案原版的 stdio 配置（仅对照，不连接）：")
print(json.dumps(STDIO_CONFIG, ensure_ascii=False, indent=2))

### 预期输出

```text
课案原版的 stdio 配置（仅对照，不连接）：
{
  "life": {
    "command": "uv",
    "args": [
      "run",
      "F:\\ProGram\\Python_Base\\Agent\\05_mcp\\tmp_nb_work\\mcp_agent_call\\server.py",
      "stdio"
    ],
    "transport": "stdio"
  }
}
```

三种传输的配置差别全在**这个 dict 的键**上：

| 传输 | 配置里给什么 | 客户端怎么连 |
|---|---|---|
| `stdio` | `command` + `args`（服务端脚本路径，**必须是绝对路径**） | 自己拉起子进程，走 stdin/stdout |
| `streamable_http` | `url` | 连已有服务（本 notebook 用的是这条） |
| `sse` | `url` | 已弃用，同 HTTP 但半双工 |

⚠️ `"args"` 里那个 `stdio` 参数**不能省**：归档服务端不传参数时会走「自检演示」
分支去抢 8000 端口，stdio 模式下服务端**绝对不能往 stdout 打印任何东西**
（协议就走在 stdout 上）。

下面是课案原版的主体。注意 `06_agent调用_langchain.py` 里其实写了**两种方式**，
但把方式 A（`create_agent`）注释掉了，直接用了方式 B（`create_deep_agent`）——
因为它更短。完整版（第 7 节）会把方式 A 拿回来讲透。

两个必须记住的点：

- **MCP 工具是异步的**：`agent.invoke()` 会失败，必须用 `await agent.ainvoke()`；
- `recursion_limit` 显式放大到 50：DeepAgents 的循环比普通 Agent 长，默认上限容易撞
  `GraphRecursionError`。

In [ ]:
async def main():
    # 0.3 版本不再支持 async with 上下文管理器，直接实例化
    client = MultiServerMCPClient(
        {
            "math_tools": {
                "url": MCP_URL,
                "transport": "streamable_http",
            },
        }
    )
    tools = await client.get_tools()
    print("MCP 工具列表：", [t.name for t in tools])

    # 方式 A：LangChain 智能体（课案原版注释掉了这一行，完整版会讲）
    # from langchain.agents import create_agent
    # agent = create_agent(model=llm, tools=tools)

    # 方式 B：DeepAgents 智能体
    agent = create_deep_agent(
        model=llm,
        tools=tools,
        system_prompt="你是生活助手，回答天气等问题时使用工具。",
    )

    # MCP 工具是异步的，必须用 ainvoke 调用智能体
    result = await agent.ainvoke(
        {"messages": [("user", "上海天气怎么样？顺便算一下 3+5")]},
        config={"recursion_limit": 50},
    )
    print("AI：", result["messages"][-1].content)


await main()

### 预期输出

```text
MCP 工具列表： ['add', 'sub', 'mul', 'div', 'get_weather']
```

**就这一行，是这一节全部的「仪式」** —— 对比第 5 节那一大段手工转换：
`get_tools()` 出来直接就是 `BaseTool` 列表，名字原样保留（`add` / `sub` / `mul` / `div`），
**没有加任何前缀**，说明适配器是「透明」的。

### 预期输出（模型自己决定调哪几个工具、怎么说）

```text
AI： 两件事都办好了：

- **上海天气**：晴，25 ℃，天气不错，适合外出。
- **3 + 5 = 8**

还需要我查其他城市的天气或算点别的吗？
```

这一次提问是**两个问题混在一句里**（「上海天气怎么样？顺便算一下 3+5」），
模型需要自己在 5 个工具里挑出 `get_weather` 和 `add`。
调了几个、先调哪个、话怎么说，**每次运行不同**，看结构就行。

⚠️ 注意这里**没有**步数校验：课案原版只靠系统提示词。
这也是完整版要补的东西 —— 见下一节。

## 7. ② LangChain · 完整版（264 行）：`create_agent` + 工具格式探测 + 步数校验

完整版把课案原版省掉的细节都补上了，三件事：

### 7.1 看一眼适配器到底转出了什么

`first.args_schema` 就是 MCP 那边 `inputSchema` 转过来的**参数模型** ——
也就是上一节我们手工拼进 `function.parameters` 的那份 JSON Schema。

⚠️ 本机实测：`args_schema` 在 **pydantic 模型**和 **普通 dict** 两种形态间会随版本变，
所以必须写 `hasattr` 分支；直接当 dict 用会 `AttributeError`。

In [ ]:
from langchain.agents import create_agent


async def inspect_tools():
    """把 get_tools() 转出来的东西摊开看一眼：名字 + 参数模型。"""
    client = MultiServerMCPClient(
        {"math_tools": {"url": MCP_URL, "transport": "streamable_http"}}
    )
    tools = await client.get_tools()
    print("获取到的 MCP 工具:", [t.name for t in tools])

    first = tools[0]
    schema = first.args_schema
    props = (
        schema.model_json_schema().get("properties", {})
        if hasattr(schema, "model_json_schema")
        else (schema or {}).get("properties", {})
    )
    print(f"  · {first.name} 的参数照旧来自 MCP 的 inputSchema：{list(props.keys())}")
    print(f"  · args_schema 的形态：{type(schema).__name__}")


await inspect_tools()

### 预期输出

```text
获取到的 MCP 工具: ['add', 'sub', 'mul', 'div', 'get_weather']
  · add 的参数照旧来自 MCP 的 inputSchema：['a', 'b']
  · args_schema 的形态：dict
```

第一行和第 6 节完全一致；第二行才是这一段的目的：
**`args_schema` 就是上一节我们手工拼进 `function.parameters` 的那份 JSON Schema**，
适配器把它**原样**搬了过来（属性名仍是 `a` / `b`）。

第三行是本机 `langchain-mcp-adapters 0.3.2` 的真实形态 —— `dict`。
而源文件注释里记的是「本机实测命中的是 pydantic 分支」：**判断代码必须两种都兼容**，
因为换一个版本的适配器，这一行打印出来的就是 `BaseModel`（`hasattr` 分支就是为它写的）。

### 7.2 `create_agent`：循环被框架吃掉了，校验只能放在外面

三行建 Agent：模型 + 工具 + 系统提示词。

⚠️ 关键差别：`create_agent` 的内部循环是「模型不再请求工具 → 图就结束」，
**我们无法在循环内部拦它**，只能在一次 `ainvoke` 返回之后检查：
表达式至少需要几步运算，就至少该发生几次工具调用。
少调了就把**整段历史带上**、补一条 `user` 消息，再 `ainvoke` 一次 ——
这等价于手工实现「让 Agent 继续干活」的外层循环。

（必须带上整段历史，否则模型会丢掉前面算出来的中间结果，从零重算。）

In [ ]:
async def main() -> None:
    client = MultiServerMCPClient(
        {
            "math_tools": {
                "url": MCP_URL,
                "transport": "streamable_http",
            },
        }
    )

    # 一行拿到全部 LangChain 工具：MCP 的 Tool 对象已经被转成 BaseTool
    tools = await client.get_tools()

    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=(
            "你是一个计算助手。必须调用工具做每一步运算，禁止心算；"
            "表达式里有几步运算就调几次工具，算完后直接给出最终数字。"
        ),
    )

    print("调用 agent...")
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "2+4*6"}]}
    )

    # 流程完整性校验（和生产实践里 05 那节同一个思路，只是写法要适配框架）
    required_steps = sum("2+4*6".count(op) for op in "+-*/")     # = 2
    for push_back in range(1, 4):        # 最多推 3 次，防止模型死活不改导致无限循环
        # tool_calls 挂在各条 AIMessage 上，逐条累加才是「实际发生了几次工具调用」
        made = sum(len(getattr(m, "tool_calls", None) or []) for m in result["messages"])
        if made >= required_steps:
            break                        # 步数够了就收工，不多问模型一次（省 token、省时间）
        print(f"⚠️  只发生了 {made} 次工具调用，2+4*6 至少需要 {required_steps} 步 —— "
              f"把历史带上再推它一次（第 {push_back} 次）。")
        result = await agent.ainvoke({
            "messages": list(result["messages"]) + [{
                "role": "user",
                "content": f"你只调用了 {made} 次工具，还有运算没算完（至少需要 {required_steps} 步）。"
                           "请继续调用工具算出剩余步骤，禁止心算，算完给出最终数字。",
            }]
        })

    # 把中间过程打出来，让「Agent 自己决定调哪个工具」这件事可见：
    # 轨迹里会出现三类消息 —— AIMessage（可能带 tool_calls）/ ToolMessage / HumanMessage
    print("\n--- 消息轨迹 ---")
    for msg in result["messages"]:
        kind = type(msg).__name__
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            for tc in tool_calls:
                print(f"  [{kind}] 调用工具 {tc['name']}，参数 {tc['args']}")
        elif kind == "ToolMessage":
            print(f"  [{kind}] 工具返回 {msg.content}")
        elif getattr(msg, "content", None):
            # 中途的 AI 说明文字也打出来（截断 80 字），能看出模型的推理顺序
            text = msg.content if isinstance(msg.content, str) else str(msg.content)
            print(f"  [{kind}] {text[:80]}")

    print("\n最终答案:")
    print(result["messages"][-1].content)


await main()

### 预期输出（轨迹里带 session 相关的 id，每次运行不同）

```text
调用 agent...

--- 消息轨迹 ---
  [HumanMessage] 2+4*6
  [AIMessage] 调用工具 mul，参数 {'a': 4, 'b': 6}
  [ToolMessage] 工具返回 [{'type': 'text', 'text': '24.0', 'id': 'lc_d725d33e-...'}]
  [AIMessage] 调用工具 add，参数 {'a': 2, 'b': 24}
  [ToolMessage] 工具返回 [{'type': 'text', 'text': '26.0', 'id': 'lc_92cb1fd1-...'}]
  [AIMessage] 2+4*6 = **26**

最终答案:
2+4*6 = **26**
```

这段轨迹把「框架内部到底发生了什么」摊开了，四点：

1. **`AIMessage` 带 `tool_calls` → `ToolMessage` 带结果**，一出一进成对出现 ——
   和原生 SDK 手写的 `messages` 是**同一个结构**，只是这里由框架自动维护；
2. **工具返回值是 content block 列表**（`[{'type': 'text', 'text': '24.0', ...}]`），
   不是字符串 —— 这正是上一节 `str(result)` 那个 `CallToolResult` 的另一种形态；
3. **`id` 每次运行不同**（session 级 UUID），所以这段只能看结构、**别逐字比对**；
4. **本次没有出现「推回去」那行警告**：模型自己就调了 `mul` + `add` 两步，
   校验分支没被触发 —— 又一次印证「护栏是兜底，不是常规路径」。

## 8. ③ DeepAgents · 完整版（314 行）：只差一行，但多出一整套工具

课案这一节只给了一行差异说明：

> 和 LangChain 版的唯一区别：`create_agent` → `create_deep_agent`，
> 额外获得文件系统、任务拆解等内置能力。

| | LangChain | DeepAgents |
|---|---|---|
| MCP 客户端 | `MultiServerMCPClient` | **完全相同** |
| 获取工具 | `client.get_tools()` | **完全相同** |
| 创建 Agent | `create_agent(model, tools)` | `create_deep_agent(model, tools)` |
| 额外能力 | 无 | 内置文件系统、任务拆解、子 Agent 委派 |

关键结论：**DeepAgents 对 MCP 没有任何特殊要求。**
因为 MCP 工具经由 `langchain-mcp-adapters` 之后已经是标准 LangChain `BaseTool`，
而 `create_deep_agent` 接受的就是 `BaseTool` 列表 ——
协议层和框架层是解耦的，MCP 在 LangChain 侧「转一次」，下游谁用都一样。

### 8.1 内置工具是自动挂上的，而且**随版本变**

`create_deep_agent` 会在你给的 tools 之外，自动挂上一整套内置工具（middleware 注入）。
本机 `deepagents 0.7.13` 实测大概是：

    ls / read_file / write_file / edit_file / delete / glob / grep   —— 文件系统（虚拟工作区）
    execute                                                          —— 在沙箱里执行 shell
    task                                                             —— 把子任务委派给子 Agent

注意 0.7.13 里**已经没有 `write_todos`** 了，任务拆解改由系统提示词里的规划规范驱动。
所以「课案表格 = 概念」，具体工具名要以你装到的版本为准 ——
这正是下面那段**自动探测**代码的用处：**不要照抄这里的清单**。

### 8.2 `deepagents` 缺包时不能炸栈

`deepagents` 属于「未必每台机器都装」的库，所以 `_load_create_deep_agent()`
用 `try/except ImportError` **延迟导入**，缺包时打印中文提示 + 安装命令并返回 `None`
（调用方见 `None` 就正常退出，不抛 traceback）。

⚠️ 一个诚实的提醒：本 notebook 第 6 节**保留了课案原版那一句顶部**
`from deepagents import create_deep_agent`（为忠实呈现原版写法），
所以本 notebook 整体**仍然要求装好 `deepagents`**。
真要在「可能缺包」的环境里跑，请统一用下面这种延迟导入的写法 ——
这也是为什么生产代码里 `import` 有时不该放在文件头。

In [ ]:
def _load_create_deep_agent():
    """缺包时不能让模块 import 就炸，要打印中文提示并降级退出。"""
    try:
        from deepagents import create_deep_agent
        return create_deep_agent
    except ImportError:
        print("❌ 未安装 deepagents，无法运行本节示例。")
        print("   安装命令： uv add deepagents")
        print("   或者直接看第 6/7 节，那两个只需要 langchain-mcp-adapters。")
        return None


def _text_of(content) -> str:
    """DeepAgents / LangChain 1.x 的 message.content 可能是 str，也可能是内容块列表。

    取最后一条回答时两种形态都要能处理，否则会打出 "[{'type': 'text', ...}]"。
    """
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
            else:
                parts.append(str(block))
        return "".join(parts)
    return str(content)


def _collect_tool_names(agent) -> set[str]:
    """从编译后的 LangGraph 图里读出「这个 Agent 能调用的全部工具名」。

    实现要点：CompiledStateGraph.nodes["tools"] 是 ToolNode，
    它的 .bound.tools_by_name 是 {工具名: BaseTool} 字典。

    不同 deepagents 版本的内部结构会变，所以这里做「能读到就读，
    读不到就返回空集合」的宽松处理 —— 探测失败绝不能中断教学演示。
    """
    names: set[str] = set()
    try:
        tool_node = agent.nodes["tools"]
        names.update(tool_node.bound.tools_by_name.keys())
    except Exception:
        pass
    return names


print("deepagents 载入结果：", "OK" if _load_create_deep_agent() else "缺包")

### 预期输出

```text
deepagents 载入结果： OK
```

打出 `OK` 说明 `try/except ImportError` 这条延迟导入的路走通了。
换成一台没装 `deepagents` 的机器，这里打印的会是三行中文提示 + `uv add deepagents`，
**而且不会抛 traceback** —— 这就是源文件刻意把 import 放进函数里的原因。

### 8.3 只改一行：`create_agent` → `create_deep_agent`

下面这段和上一节**逐行对照**看：

- ①② 建 client、拿工具 —— 和上一节**一字不差**；
- ③ 唯一的区别在 `create_deep_agent(` 这一行；
- ④ 后面多了两样：把内置工具**自动列出来**，以及轨迹里标出每次调用是「MCP」还是「内置」。

把「来自 MCP 的」和「内置的」分成两组打印，是这一节最有价值的一段 ——
工具列表变长之后，读者要能看出**模型调的是谁**。

In [ ]:
async def main(create_deep_agent) -> None:
    client = MultiServerMCPClient(
        {
            "math_tools": {
                "url": MCP_URL,
                "transport": "streamable_http",
            }
        }
    )

    tools = await client.get_tools()
    print("获取到的 MCP 工具:", [t.name for t in tools])

    agent = create_deep_agent(
        model=llm,
        tools=tools,
        system_prompt=(
            "你是一个计算助手。必须调用工具做每一步运算，禁止心算；"
            "表达式里有几步运算就调几次工具，算完后直接给出最终数字。"
        ),
    )

    # DeepAgents 到底比普通 Agent 多了哪些工具？直接查编译后的图：
    # 用集合差集把「你给的」和「它自带的」分开，版本升级导致清单变化也能一眼看出。
    mcp_names = {t.name for t in tools}
    all_names = _collect_tool_names(agent)
    builtin = sorted(all_names - mcp_names)
    print(f"\nAgent 可用的全部工具 {len(all_names)} 个：")
    print(f"  · 来自 MCP（你提供的）：{sorted(mcp_names)}")
    print(f"  · 内置（DeepAgents 自动挂上的）：{builtin}")

    print("\n调用 agent...（DeepAgents 会先拆任务，比普通 Agent 多几轮，请稍等）")
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "2+4*6"}]},
        config={"recursion_limit": 50},
    )

    # 流程完整性校验：和上一节完全一样的写法，说明这个套路与框架无关
    required_steps = sum("2+4*6".count(op) for op in "+-*/")     # = 2
    for push_back in range(1, 4):        # 最多推 3 次，避免模型死活不改导致无限循环
        made = sum(len(getattr(m, "tool_calls", None) or []) for m in result["messages"])
        if made >= required_steps:
            break                        # 步数达标就收工，不再多问模型一次
        print(f"⚠️  只发生了 {made} 次工具调用，2+4*6 至少需要 {required_steps} 步 —— "
              f"把历史带上再推它一次（第 {push_back} 次）。")
        # 重推时 config 必须一起传：recursion_limit 不会自动继承上一次调用
        result = await agent.ainvoke(
            {
                "messages": list(result["messages"]) + [{
                    "role": "user",
                    "content": f"你只调用了 {made} 次工具，还有运算没算完"
                               f"（至少需要 {required_steps} 步）。"
                               "请继续调用工具算出剩余步骤，禁止心算，算完给出最终数字。",
                }]
            },
            config={"recursion_limit": 50},
        )

    # 打印轨迹时把每次调用标上「MCP」或「内置」——
    # 这是 DeepAgents 最容易被误解的地方：工具列表变长之后，要看得出模型调的是谁
    print("\n--- 消息轨迹（注意有没有内置工具出现） ---")
    for msg in result["messages"]:
        kind = type(msg).__name__
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            for tc in tool_calls:
                # mcp_names 是「你通过 MultiServerMCPClient 提供的那些工具名」的集合
                origin = "MCP" if tc["name"] in mcp_names else "内置"
                print(f"  [{kind}][{origin}] 调用 {tc['name']}，参数 {str(tc['args'])[:90]}")
        elif kind == "ToolMessage":
            print(f"  [{kind}] 返回 {_text_of(msg.content)[:90]}")
        elif getattr(msg, "content", None):
            print(f"  [{kind}] {_text_of(msg.content)[:90]}")

    print("\n最终答案:")
    print(_text_of(result["messages"][-1].content))


# 源文件这一段的原文是：
#     create_deep_agent = _load_create_deep_agent()
#     if create_deep_agent is None:
#         sys.exit(0)                       # 缺包时正常退出，不抛 traceback
# notebook 里 sys.exit() 会把**内核一起退掉**，所以那一行只在「导出成 .py 单跑」时才走 ——
# 同一个 `if "__file__" in globals():` 守卫，和前面 asyncio.run 的处理保持一致。
create_deep_agent = _load_create_deep_agent()
if create_deep_agent is None:
    if "__file__" in globals():
        sys.exit(0)                       # 缺包时正常退出，不抛 traceback
    print("缺包，本节跳过 —— notebook 里不能 sys.exit()，那会把内核一起退掉。")
if create_deep_agent is not None:
    await main(create_deep_agent)

### 预期输出（内置工具清单随 deepagents 版本会变，别逐字比对）

```text
获取到的 MCP 工具: ['add', 'sub', 'mul', 'div', 'get_weather']

Agent 可用的全部工具 14 个：
  · 来自 MCP（你提供的）：['add', 'div', 'get_weather', 'mul', 'sub']
  · 内置（DeepAgents 自动挂上的）：['delete', 'edit_file', 'execute', 'glob', 'grep',
    'ls', 'read_file', 'task', 'write_file']

调用 agent...（DeepAgents 会先拆任务，比普通 Agent 多几轮，请稍等）

--- 消息轨迹（注意有没有内置工具出现） ---
  [HumanMessage] 2+4*6
  [AIMessage][MCP] 调用 mul，参数 {'a': 4, 'b': 6}
  [ToolMessage] 返回 24.0
  [AIMessage][MCP] 调用 add，参数 {'a': 2, 'b': 24}
  [ToolMessage] 返回 26.0
  [AIMessage] 2+4*6 = 26

最终答案:
2+4*6 = 26
```

**本机 `deepagents 0.7.13` 实测：5 个 MCP 工具 + 9 个内置工具 = 14 个。**
这一串数字和工具名是**版本相关**的（本课文件头也专门提醒过：不要照抄清单），
所以这段只核对结构：

| 看到的 | 说明 |
|---|---|
| `来自 MCP（你提供的）` 5 个 | 就是你 `MultiServerMCPClient` 那个 dict 里的服务暴露的全部工具 |
| `内置（DeepAgents 自动挂上的）` 9 个 | `middleware` 注入，**你一行都没写**；本次**没有** `write_todos`（0.7.13 已移除） |
| 轨迹里的 `[MCP]` 标记 | 两次调用都落在 MCP 工具上 —— 简单算术题**根本用不到**内置工具 |
| `[ToolMessage] 返回 24.0` | 注意这里**不再是** content block 列表：`_text_of()` 已经把它取成纯文本 |

**最后一行是本课最想让你记住的对比**：同一条 `2+4*6`，
普通 `create_agent`（第 7 节）和 `create_deep_agent`（这一节）
**都只调了 `mul` + `add` 两步**，内置工具一个没用上。
但后者的 tool schema 长了 9 个工具 —— 这就是「简单任务用 `create_agent` 就够了」的实测依据。

## 9. 三档横向对照：到底差在哪

把三种方式放在同一张表里（**本 notebook 的结论**）：

| | ① 原生 OpenAI SDK | ② LangChain | ③ DeepAgents |
|---|---|---|---|
| 建连接 | `Client(transport)` / `Client(url)` | `MultiServerMCPClient({...})` | **同 ②** |
| 工具格式转换 | **手写** `mcp_tool_to_openai()` | `await client.get_tools()` | **同 ②** |
| 建 Agent | 无（自己写循环） | `create_agent(model=llm, tools=tools)` | `create_deep_agent(model=llm, tools=tools)` |
| 跑循环 | **手写** `for` + `tool_calls` | 框架内部 | 框架内部 |
| 流程完整性校验 | 循环**内部**拦（`continue` 推回去） | 只能 `ainvoke` **返回后**检查再推 | 同 ② |
| 内置工具 | 无 | 无 | 文件系统 / 任务拆解 / 子 Agent |
| 本次实测调用轮次 | 2 轮（1 次工具 + 1 次收尾） | 1~2 轮 | 通常多于 ② |

### 9.1 三句话记住

1. **转换那一步是分水岭**：原生 SDK 里它是你手写的 3 行；适配器里它是 `get_tools()`
   一行 —— 但它做的事**完全一样**（名字 / 描述 / inputSchema → BaseTool）；
2. **循环是第二个分水岭**：手写循环你能在**内部**拦模型（推回去补齐）；
   框架的循环你只能在**外面**补一次 `ainvoke` —— 所以「流程完整性校验」的写法
   在 ① 和 ②③ 里长得不一样，但思路一致；
3. **DeepAgents 对 MCP 没有特殊要求**：`MultiServerMCPClient` 那两行一字未改。
   多出来的是**内置工具**，代价是每轮 tool schema 更长、**token 消耗明显更高** ——
   简单任务用 `create_agent` 就够了。

## 小结

- **MCP 的价值**是「工具一次编写、处处可用」；**适配器的价值**是「把可用变成好用」；
- `MultiServerMCPClient(...).get_tools()` 一行顶掉原生 SDK 里的手工转换，
  转出来的 `args_schema` 就是 MCP 的 `inputSchema`；
- 工具循环：原生 SDK 手写、LangChain / DeepAgents 由框架内部完成；
- `create_deep_agent` 会在你给的工具之外**自动挂上一整套内置工具**（版本相关，要探测不要背）；
- `create_agent` → `create_deep_agent` 只有一行之差，因为协议层与框架层已经解耦；
- **流程完整性校验（guardrail）** 是本课最值得带走的生产经验：
  模型「少调工具」是真实存在的，靠系统提示词压不住，要靠代码算下界 + 检查 + 推回去。

## 常见坑

1. ★ **别在 notebook 进程里 `import` 归档脚本**（本机实测，踩过两次）：
   归档脚本是按「**独立可执行**」写的，文件头就有
   `sys.stdout.reconfigure(encoding="utf-8")`。命令行/子进程里没事，
   但被内核 in-process 加载时 `sys.stdout` 是 IPython 的 `OutStream`，没有这个方法 →
   `AttributeError: 'OutStream' object has no attribute 'reconfigure'`
   （报错位置看着像归档文件第 47 行，根因却在「不该 import 它」）。
   同一族的第二种表现：mcp 的 stdio 客户端把 `sys.stderr` 当**默认参数**
   在**导入时**就绑给了 subprocess，而 `OutStream` 没有真 `fileno()` →
   `RuntimeError: Client failed to connect: fileno`。
   **正确做法：归档脚本只当子进程跑（本 notebook 的 `server.py` 就是被 `Popen` 起来的）。**
2. **Jupyter 内核里 stdio 传输连不上**（本 notebook 实测）：
   Windows 的 Jupyter 内核跑 `_WindowsSelectorEventLoop`，而它
   **不实现 asyncio 的子进程能力** ——
   `asyncio.create_subprocess_exec` 直接抛 `NotImplementedError`，
   `StdioTransport` 报 `RuntimeError: Client failed to connect: fileno`。
   所以 notebook 里的 MCP 客户端一律走 **HTTP**。
3. **stdio 传输 + `create_deep_agent` 会卡死**（`02_langchain/21_MCP进阶_官方补充.py` 实测）：
   同样一个 stdio MCP 工具，交给 `create_agent` 秒回，交给 `create_deep_agent`
   **240 秒不返回**（起两次都一样）；换成 `streamable_http` 立刻正常。
   **结论：DeepAgents 接 MCP 用 HTTP 传输** —— 本 notebook 与课案 07 用的都是 HTTP。
4. **`async with MultiServerMCPClient(...)` 会抛 `NotImplementedError`**
   （`langchain-mcp-adapters ≥ 0.1.0`）：正确写法是**直接实例化**再 `await client.get_tools()`。
5. **`args_schema` 的形态会随版本变**：可能是 pydantic 模型（有 `model_json_schema()`），
   也可能是普通 dict —— 必须写 `hasattr` 分支，直接当 dict 用会 `AttributeError`。
6. **MCP 工具是异步的**：`agent.invoke()` 会失败，必须 `await agent.ainvoke()`。
7. **`agent.ainvoke` 时 `recursion_limit` 不会自动继承**：重推一次就得再传一遍，
   否则会撞 `GraphRecursionError`。
8. **原生 SDK 里工具结果的形状要你自己处理**：`str(result)` 是
   `CallToolResult(content=[TextContent(...)])`，不是纯文本 —— 要取
   `result.content[0].text`。
9. **`role="tool"` 的消息必须带 `tool_call_id`**，且和上一条 assistant 的
   `tool_calls` **成对出现**，否则模型接口直接 400。
10. **`model_dump()` 而不是手写 dict**：SDK 的新字段（`refusal` / `annotations`）它一并带上，
    手写容易漏字段导致下一轮 400。
11. **`mcp.run(...)` 会永久阻塞内核**：常驻服务必须起独立进程（本 notebook 用
    `subprocess.Popen` + 轮询端口就绪 + 末尾 `taskkill /F /T`）。
12. **回环被代理拦**：连本机服务前把 `127.0.0.1,localhost` 加进 `NO_PROXY`。
13. **端口冲突**：本课用 **8130**。课案的 8000 被同章其它 notebook 占着，
    跑之前先看 `_port_in_use()` 的提示。
14. ★ **「端口通了」不等于「连的是自己的服务」**（本机并发跑时实测踩到）：
    同章 5 个 notebook 并发时，只要有一个抢了同一个端口，本 notebook 的可就绪探测
    仍会返回 True（它只知道「有人监听」），但连上去的工具清单对不上，
    表现成一句含糊的 `RuntimeError: Client failed to connect: `。
    **解法就是第 3 节那道「身份核对」**：把 `list_tools()` 的结果和期望的
    `EXPECTED_TOOLS` 比一比 —— 对不上就立刻说清是端口被抢，而不是让人去猜。
    同理，`subprocess.Popen` 起服务时**要把子进程的 stdout 接住**：
    它抢不到端口会立刻退出并打印原因，读出来比任何猜测都快。

## 官方链接

- MCP 协议官网（规范 + 传输方式）：<https://modelcontextprotocol.io/>
- FastMCP 客户端/服务端文档：<https://gofastmcp.com/>
- langchain-mcp-adapters（`MultiServerMCPClient`）：<https://github.com/langchain-ai/langchain-mcp-adapters>
- LangChain `create_agent`：<https://docs.langchain.com/oss/python/langchain/agents>
- DeepAgents 总览：<https://docs.langchain.com/oss/python/deepagents/overview>
- OpenAI Function Calling 指南：<https://platform.openai.com/docs/guides/function-calling>

## 最后：关掉服务

模板第 6 节第 5 条要求：常驻服务必须在**最后一个 cell** 里关掉。
Windows 上要**连子进程树一起收**（`/T`）—— `server.py` 自己还会拉起别的进程时，
只杀父进程会留下孤儿继续占端口。

顺带演示两个工程细节：

- **`WinError 32`**：刚被 `taskkill` 的进程文件句柄不会立刻释放，
  删临时目录要**重试 + 容忍失败**，不能因为删不掉就报错；
- 关完再 `_port_in_use()` 复查一次，确认端口真的空出来了。

> 下一格是**本 notebook 的最后一个 cell**（最后执行的一格）：关服务、
> 复查端口、清理临时目录，三件事一次做完。

### 预期输出（最后一格跑完你会看到）

```text
已关闭 MCP 服务端（PID 50392）。
端口已释放： True
临时目录已清理： True
```

三行分别对应三件事：

| 行 | 含义 |
|---|---|
| `已关闭 MCP 服务端（PID …）` | `taskkill /F /T` 真的打到了那个子进程（`PID` 每次都不同） |
| `端口已释放： True` | 8130 空出来了 —— **这一条比上一行更重要**，端口没释放等于服务还在 |
| `临时目录已清理： True` | `server.py` 连同子目录一起删掉了（删失败会返回 `False`，但**不让整个 notebook 报错**） |

想自己复核「真的没留孤儿进程」，在终端里跑：

```powershell
Get-NetTCPConnection -LocalPort 8130 -State Listen -ErrorAction SilentlyContinue
```

没有任何输出，就说明 8130 干净了。

In [ ]:
if server.poll() is None:
    subprocess.run(["taskkill", "/F", "/T", "/PID", str(server.pid)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    print(f"已关闭 MCP 服务端（PID {server.pid}）。")
else:
    print("MCP 服务端已自行退出。")

# 等端口释放，再复查一次（最多 5 秒）
for _ in range(20):
    if not _port_in_use(HTTP_HOST, HTTP_PORT):
        break
    time.sleep(0.25)
print("端口已释放：", not _port_in_use(HTTP_HOST, HTTP_PORT))


def remove_temp_dir(path):
    """删临时目录：Windows 上刚被 taskkill 的进程句柄未释放会 WinError 32，重试 + 容忍失败。"""
    import shutil

    for _ in range(5):
        try:
            shutil.rmtree(path)
            return True
        except FileNotFoundError:
            return True
        except OSError:
            time.sleep(0.5)
    return False


print("临时目录已清理：", remove_temp_dir(SERVER_DIR))